# S1 — 진단 run: 역전 + proxy 예측

**목적:** 단일층 `R_l(t)`를 전 층 측정 → (RQ2) 단기 best ≠ 수렴 best 역전이 있나, (RQ4) HVP proxy가 회복을 예측하나, (RQ6) 출력변화 작은 층이 더 낫나.

**규모:** 비트 {8,4,2} × 전 층(21) × seed{0,1,2} × t{30,100,300,plateau}.

**산출 그림(Drive figures/):** R_l(t) 궤적 · 역전 산점도 · proxy↔회복 산점도 · ρ vs t 교차 · 갭/역전강도 vs 비트.

엔진(`qat_engine.py`)이 모든 로직, 이 노트북은 sweep·플롯만. (플롯 라벨은 폰트 안전 위해 영문.)

In [ ]:
# --- Colab 셋업 ---
import os
REPO = '/content/26_Capstone'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/u-nsiq/26_Capstone.git {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(REPO)
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- 엔진 + Drive + 경로 ---
from qat_engine import *
import numpy as np, matplotlib.pyplot as plt

try:
    from google.colab import drive; drive.mount('/content/drive')
    ART = '/content/drive/MyDrive/26_Capstone'
except Exception:
    ART = './_local_art'
for sub in ['checkpoints','outputs','figures']:
    os.makedirs(f'{ART}/{sub}', exist_ok=True)
DATA_ROOT = f'{ART}/data'
CKPT      = f'{ART}/checkpoints/resnet18_cifar100_fp32.pt'
OUT       = f'{ART}/outputs/s1_runs.jsonl'
FIG       = f'{ART}/figures'
print('device', DEVICE, '| ART', ART)

In [ ]:
# --- S1 설정 ---
BITS          = [8, 4, 2]   # 낮은끝 앵커 / 주력 / 큰갭(breakdown 가능)
SEEDS         = [0, 1, 2]
T_EVAL        = (30, 100, 300)
LR            = 1e-3
N_BATCH_PROXY = 4           # proxy 추정용 calib 배치 수
print('bits', BITS, '| seeds', SEEDS, '| t', T_EVAL, '+plateau')

## baseline + 층 목록 + 비용

In [ ]:
train_loader, val_loader, calib_loader = get_loaders('cifar100', batch=128, calib_size=512, data_root=DATA_ROOT)
assert os.path.exists(CKPT), 'FP32 체크포인트 없음 — S0를 먼저 (baseline 학습→캐시)'
fp_model = load_model('resnet18', 'cifar100', ckpt=CKPT)
fp_acc = evaluate(fp_model, val_loader)
layers = list_quant_layers(make_ptq_model(fp_model, 4))
costs  = get_layer_costs(make_ptq_model(fp_model, 4), layers)
print(f'FP32 천장 {fp_acc:.2f} | 양자화 층 {len(layers)}개')

## 스모크 — 전체 돌리기 전 3층·1seed로 기계 점검 (~몇 분)
여기서 에러나면 엔진에서 고치고 `git pull` 후 재실행. 통과하면 아래 full sweep로.

In [ ]:
# 대표 3층(이른/중간/늦은) x 1 seed x W4 로 파이프라인 점검
sm_layers = [layers[1], layers[len(layers)//2], layers[-1]]
pm = make_ptq_model(fp_model, 4)
base = evaluate(pm, val_loader)
print(f'W4 PTQ {base:.2f} (gap {fp_acc-base:.2f}%p)')
px = proxy_scores(pm, fp_model, 4, calib_loader, layers=sm_layers, n_batches=N_BATCH_PROXY)
for L in sm_layers:
    m = make_ptq_model(fp_model, 4); b = evaluate(m, val_loader)
    set_trainable(m, [L])
    r = short_qat(m, train_loader, val_loader, eval_at=T_EVAL, plateau=True, seed=0, lr=LR, momentum=0.0)
    rec = {t: round(r[t]-b, 3) for t in r}
    nh = px[L]['normHd2']; dt = px[L]['dtHd']
    print(f'{L}  R(t)={rec}  normHd2={nh:.2e}  dtHd={dt:.2e}')
print('스모크 OK — full sweep로')

## 전체 sweep (비트 × 전층 × seed) — 몇 시간, Drive에 증분 저장
중간에 끊겨도 `s1_runs.jsonl`에 개별 run이 쌓임. (분석은 메모리 RESULTS 기준이라 한 세션에서 sweep→분석 권장.)

In [ ]:
RESULTS = {}
import json as _json
for bit in BITS:
    pm = make_ptq_model(fp_model, bit)
    ptq_acc = evaluate(pm, val_loader); gap = fp_acc - ptq_acc
    proxies = proxy_scores(pm, fp_model, bit, calib_loader, layers=layers, n_batches=N_BATCH_PROXY)
    _json.dump(proxies, open(f'{ART}/outputs/s1_proxies_w{bit}.json', 'w'))   # proxy 영구 저장 (Codex #1)
    R = {L: {} for L in layers}
    for L in layers:
        for s in SEEDS:
            m = make_ptq_model(fp_model, bit); b = evaluate(m, val_loader)
            set_trainable(m, [L])
            r = short_qat(m, train_loader, val_loader, eval_at=T_EVAL, plateau=True, seed=s, lr=LR, momentum=0.0)
            for t in r:
                R[L].setdefault(t, []).append(r[t] - b)
            log_run({'phase':'S1','bit':bit,'layer':L,'seed':s}, {str(t): r[t]-b for t in r}, path=OUT)
    RESULTS[bit] = dict(gap=gap, ptq_acc=ptq_acc, proxies=proxies, R=R)
    print(f'[bit {bit}] gap {gap:.2f}%p — {len(layers)}층 x {len(SEEDS)}seed 완료, proxy 저장됨')
print('sweep 완료')

## 분석 + 그림 (Drive figures/에 PNG 저장)

In [ ]:
# 헬퍼: seed 평균/표준편차 회복 · proxy 벡터 · 신호 검증 (claude.ai)
def mean_R(bit, t):
    return [float(np.mean(RESULTS[bit]['R'][L][t])) for L in layers]
def std_R(bit, t):
    return [float(np.std(RESULTS[bit]['R'][L][t])) for L in layers]
def proxy_vec(bit, key):
    return [RESULTS[bit]['proxies'][L][key] for L in layers]

NOISE_FLOOR = 0.01   # S0 단일층 recovery seed-std (%p) — spread 비교 기준선

def signal_check(bit, t):
    # 역전이 신호인지 noise인지: spread(층간 퍼짐)/noise(평균 seed-std)=SNR, seed간 순위 일치=rank_stability
    means = mean_R(bit, t); seed_noise = std_R(bit, t)
    spread = float(np.std(means)); noise = float(np.mean(seed_noise))
    per_seed = [[RESULTS[bit]['R'][L][t][s] for L in layers] for s in range(len(SEEDS))]
    pairs = [spearman(per_seed[i], per_seed[j]) for i in range(len(SEEDS)) for j in range(i+1, len(SEEDS))]
    rank_stab = float(np.nanmean(pairs)) if pairs else float('nan')
    return dict(spread=spread, seed_noise=noise, snr=spread/max(noise,1e-9),
                rank_stability=rank_stab, mean_recovery=float(np.mean(means)))

def is_breakdown(bit):
    return RESULTS[bit]['ptq_acc'] < 5.0   # CIFAR-100 random ~1% → 붕괴 판정
print('헬퍼 준비: mean_R/std_R/signal_check/is_breakdown')

In [ ]:
# FIG0 — 신호 검증 (★ 역전 믿기 전 필수): 층별 회복 mean±std(seed) + SNR/순위안정성
chk_bits = [b for b in BITS if RESULTS[b]['gap'] > 0.5]
fig, axs = plt.subplots(len(chk_bits), 2, figsize=(12, 4*len(chk_bits)), squeeze=False)
for r, bit in enumerate(chk_bits):
    for c, t in enumerate([30, 'plateau']):
        m = np.array(mean_R(bit, t)); sd = np.array(std_R(bit, t)); order = np.argsort(m)[::-1]
        ck = signal_check(bit, t); sn = ck['snr']; rs = ck['rank_stability']
        ax = axs[r][c]
        ax.errorbar(range(len(layers)), m[order], yerr=sd[order], fmt='o', ms=3, capsize=2)
        ax.axhline(NOISE_FLOOR, ls=':', color='red', lw=1)
        ax.set_xlabel('layer (sorted by recovery)'); ax.set_ylabel('recovery (%p)')
        ax.set_title(f'W{bit} t={t}   SNR={sn:.1f}   rank-stab={rs:.2f}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_signal_check.png', dpi=120); plt.show()
print('판정: 에러바가 서로 떨어지고 SNR>~3 & rank-stab>~0.5 여야 역전이 신호. 빨간점선=S0 노이즈')

In [ ]:
# FIG2 — 역전 산점도 (단기 vs 수렴 순위), 갭 있는 비트만. rank 1=best. 제목에 신뢰도(SNR/안정성/붕괴) 표시
from scipy.stats import rankdata
def best_rank(v):
    return len(v) + 1 - rankdata(v)
inv_bits = [b for b in BITS if RESULTS[b]['gap'] > 0.5]
fig, axs = plt.subplots(1, len(inv_bits), figsize=(5*len(inv_bits),4.5), squeeze=False)
for c, bit in enumerate(inv_bits):
    short = mean_R(bit, 30); plat = mean_R(bit, 'plateau')
    inv = inversion_strength(short, plat); ck = signal_check(bit, 30)
    sn = ck['snr']; rs = ck['rank_stability']
    tag = 'BREAKDOWN' if is_breakdown(bit) else ('noise?' if (sn < 3 or rs < 0.5) else 'ok')
    ax = axs[0][c]; ax.scatter(best_rank(short), best_rank(plat), s=25)
    lim = [0, len(layers)+1]; ax.plot(lim, lim, '--', color='gray', lw=1)
    ax.set_xlabel('short (t=30) rank (1=best)'); ax.set_ylabel('converged (plateau) rank (1=best)')
    ax.set_title(f'W{bit}  inv={inv:.2f}  [SNR={sn:.1f} stab={rs:.2f} {tag}]')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_inversion.png', dpi=120); plt.show()

In [ ]:
# FIG2 — 역전 산점도 (단기 순위 vs 수렴 순위), 갭 있는 비트만. rank 1 = best recovery (Codex)
from scipy.stats import rankdata
def best_rank(v):
    return len(v) + 1 - rankdata(v)   # 큰 회복 = rank 1 (발표 직관)
inv_bits = [b for b in BITS if RESULTS[b]['gap'] > 0.5]
fig, axs = plt.subplots(1, len(inv_bits), figsize=(5*len(inv_bits),4.5), squeeze=False)
for c, bit in enumerate(inv_bits):
    short = mean_R(bit, 30); plat = mean_R(bit, 'plateau')
    inv = inversion_strength(short, plat)
    ax = axs[0][c]; ax.scatter(best_rank(short), best_rank(plat), s=25)
    lim = [0, len(layers)+1]; ax.plot(lim, lim, '--', color='gray', lw=1)
    ax.set_xlabel('short-budget (t=30) rank  (1=best)'); ax.set_ylabel('converged (plateau) rank  (1=best)')
    ax.set_title(f'W{bit}   inversion = {inv:.2f}  (1-rho, 0..2)')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_inversion.png', dpi=120); plt.show()

In [ ]:
# FIG3 — proxy ↔ 회복 산점도 (W4): 단기 proxy vs R(30), 수렴 proxy vs R(plateau)
bit = 4
fig, axs = plt.subplots(1, 2, figsize=(11,4.5))
sh = mean_R(bit,30); nh = proxy_vec(bit,'normHd2')
axs[0].scatter(nh, sh, s=25); axs[0].set_xscale('log')
axs[0].set_xlabel('proxy ||H d||^2  (short)'); axs[0].set_ylabel('R(t=30)')
axs[0].set_title(f'short proxy   rho = {spearman(nh,sh):.2f}')
pl = mean_R(bit,'plateau'); dh = proxy_vec(bit,'dtHd')
axs[1].scatter(dh, pl, s=25)
axs[1].set_xlabel('proxy dT H d  (converged)'); axs[1].set_ylabel('R(plateau)')
axs[1].set_title(f'converged proxy   rho = {spearman(dh,pl):.2f}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_proxy_scatter_w{bit}.png', dpi=120); plt.show()

In [ ]:
# FIG4 — rho vs t 교차 (W4): 어느 proxy가 언제 이기나
bit = 4
nh = proxy_vec(bit,'normHd2'); dh = proxy_vec(bit,'dtHd')
tk = [30,100,300,'plateau']; xs = [30,100,300,1000]
rho_short = [spearman(nh, mean_R(bit,t)) for t in tk]
rho_conv  = [spearman(dh, mean_R(bit,t)) for t in tk]
plt.figure(figsize=(7,4.5))
plt.plot(xs, rho_short, marker='o', label='||H d||^2  (short proxy)')
plt.plot(xs, rho_conv,  marker='s', label='dT H d  (converged proxy)')
plt.xscale('log'); plt.xlabel('step t  (1000 = plateau)'); plt.ylabel('Spearman rho (proxy vs recovery)')
plt.title(f'W{bit}: which proxy wins when'); plt.legend()
plt.tight_layout(); plt.savefig(f'{FIG}/s1_rho_vs_t_w{bit}.png', dpi=120); plt.show()

In [ ]:
# 요약 저장 — summary.json + csv + 신호검증 표 (Codex 보존 + claude.ai/Codex 검증)
import json as _json, csv as _csv
def _safe(v):
    return None if (v is None or (isinstance(v, float) and np.isnan(v))) else round(float(v), 4)
summary = {'fp_acc': fp_acc, 'bits': {}}
for bit in RESULTS:
    g = RESULTS[bit]['gap']
    c30 = signal_check(bit, 30); cpl = signal_check(bit, 'plateau')
    inv = inversion_strength(mean_R(bit,30), mean_R(bit,'plateau')) if g > 0.5 else None
    valid = bool(g > 0.5 and not is_breakdown(bit)
                 and c30['snr'] > 3 and cpl['snr'] > 3
                 and c30['rank_stability'] > 0.5 and cpl['rank_stability'] > 0.5)
    summary['bits'][str(bit)] = dict(
        gap=round(g,3), ptq_acc=round(RESULTS[bit]['ptq_acc'],2),
        mean_recovery_plateau=round(cpl['mean_recovery'],3),
        spread_t30=_safe(c30['spread']), spread_plateau=_safe(cpl['spread']),
        seed_noise_t30=_safe(c30['seed_noise']), seed_noise_plateau=_safe(cpl['seed_noise']),
        snr_t30=_safe(c30['snr']), snr_plateau=_safe(cpl['snr']),
        rank_stability_t30=_safe(c30['rank_stability']), rank_stability_plateau=_safe(cpl['rank_stability']),
        breakdown=bool(is_breakdown(bit)),
        inversion_strength=_safe(inv),
        inversion_trustworthy=valid,
        rho_short_R30=_safe(spearman(proxy_vec(bit,'normHd2'), mean_R(bit,30))),
        rho_conv_Rplateau=_safe(spearman(proxy_vec(bit,'dtHd'), mean_R(bit,'plateau'))),
    )
    s30 = std_R(bit,30); spl = std_R(bit,'plateau')
    with open(f'{ART}/outputs/s1_recovery_mean_w{bit}.csv', 'w', newline='') as f:
        w = _csv.writer(f); w.writerow(['layer','R30','R100','R300','Rplateau','std30','stdPlateau'])
        for i, L in enumerate(layers):
            w.writerow([L] + [round(np.mean(RESULTS[bit]['R'][L][t]),4) for t in [30,100,300,'plateau']] + [round(s30[i],4), round(spl[i],4)])
_json.dump(summary, open(f'{ART}/outputs/s1_summary.json','w'), indent=2, ensure_ascii=False)
print('저장: s1_summary.json (spread/noise/snr/rank-stab/trust/breakdown) + csv')
for b in summary['bits']:
    s = summary['bits'][b]
    print('W'+b, '| gap', s['gap'], '| spread30', s['spread_t30'], '| snr30', s['snr_t30'], '| stab30', s['rank_stability_t30'], '| inv', s['inversion_strength'], '| TRUST', s['inversion_trustworthy'], '| breakdown', s['breakdown'])

## S1 완료 — 무엇을 볼까 (★ 신호 검증 먼저)

**0. FIG0 신호 검증 (가장 먼저):** 층별 회복 에러바가 서로 *떨어져* 있고 **SNR > ~3 & rank-stab > ~0.5** 여야 순위가 진짜. 이게 깨지면 회복이 노이즈 폭 안이라 **아래 역전 숫자는 의미 없음**(랜덤 순위가 inversion≈1로 *위장*). `summary.json`의 `inversion_trustworthy=False`인 비트는 역전을 믿지 말 것.

1. **역전 산점도(FIG2):** 제목에 `[SNR / stab / ok|noise?|BREAKDOWN]` 표시됨. `ok`인 비트만 — 대각선 벗어남 = 단기 best ≠ 수렴 best.
2. **proxy 산점도 + rho vs t(FIG3·4):** `||Hd||²`가 작은 t, `dT H d`가 큰 t에서 더 높으면 → 이론(λ² vs λ) 확인.
3. **vs 비트(FIG5):** 비트 낮을수록 역전 커지나(RQ3). **단 W2가 `breakdown=True`면 그 "역전"은 *붕괴*지 역전 아님 — RQ3 추세로 보고 금지** (claude.ai 경고).

→ 결과(특히 `summary.json`의 `inversion_trustworthy` / `breakdown`) 갖고 와서 같이 해석.

## S1 완료 — 무엇을 볼까
1. **역전 산점도**: 점이 대각선에서 벗어날수록 역전 (단기 best ≠ 수렴 best). inversion strength 클수록 강함.
2. **proxy 산점도 + rho vs t**: `||Hd||^2`가 *작은 t*에서, `dT H d`가 *큰 t*에서 더 높으면 → 이론(λ² vs λ) 확인.
3. **vs 비트**: 비트 낮을수록 역전강도 커지나 (RQ3).

결과 보고 → S2(전략 run) 또는 발표 빌드. 해석은 같이.